# Module 21: Advanced Model Evaluation, Diagnostics & Hyperparameter Optimization
## Master Guide: Metrics, Learning Curves, Production Pipelines, and Optuna

In real-world machine learning, building a model is only 20% of the battle. The remaining 80% is **evaluating correctly, diagnosing bugs, preventing data leakage, and optimizing hyperparameters**.

A model with 98% accuracy can be utterly useless in production if evaluated on an imbalanced dataset!

---

### 🎯 Key Learning Objectives in this Module:
1. **The Comprehensive Evaluation Metrics Taxonomy**:
   - Confusion Matrix (TP, FP, TN, FN)
   - Precision, Recall, Specificity, F1-Score, F-Beta ($F_{0.5}, F_2$)
   - ROC-AUC vs PR-AUC (Precision-Recall Curve): When and why PR-AUC is strictly required for class imbalance
   - Brier Score & Probability Calibration (Platt Scaling, Isotonic Regression)
2. **Model Diagnostics**:
   - Learning Curves: Diagnosing High Bias (Underfitting) vs High Variance (Overfitting)
   - Validation Curves: Tracking individual hyperparameter sensitivity
3. **Leak-Free Production Pipelines with Scikit-Learn**:
   - `ColumnTransformer` (Numerical imputing/scaling + Categorical one-hot/ordinal encoding)
   - Building automated `Pipeline` objects that prevent test data leakage during Cross-Validation
4. **Modern Hyperparameter Optimization**:
   - Grid Search vs Random Search
   - Bayesian Optimization with **Optuna** (Tree-structured Parzen Estimator / TPE)
   - Pruning unpromising trials and parameter visualization
5. **Self-Assessment Practice Exercises & Detailed Solutions**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score, brier_score_loss
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
print("Evaluation & Diagnostics environment initialized!")


---
## 1. The Metric Trap: Why Accuracy is Dangerous

Suppose you are building a Fraud Detection model where only **1%** of transactions are fraudulent (990 normal, 10 fraud).
A dummy model that predicts "Normal" for **every single transaction** achieves **99.0% Accuracy**! Yet it catches **zero** fraud.

### The Diagnostic Matrix:
```text
                     ACTUAL POSITIVE (1)      ACTUAL NEGATIVE (0)
PREDICTED POS (1)    True Positive (TP)       False Positive (FP) [Type I Error]
PREDICTED NEG (0)    False Negative (FN)      True Negative (TN)  [Type II Error]
                     [Type II Error]
```

### Core Formulas:
- $\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$: *"When the model predicts Fraud, how often is it right?"*
- $\text{Recall (Sensitivity)} = \frac{\text{TP}}{\text{TP} + \text{FN}}$: *"What fraction of actual Fraud cases did the model catch?"*
- $\text{Specificity} = \frac{\text{TN}}{\text{TN} + \text{FP}}$: *"What fraction of normal transactions were correctly cleared?"*
- $F_1\text{-Score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$: Harmonic mean of Precision and Recall.
- $F_\beta\text{-Score} = (1 + \beta^2) \frac{\text{Precision} \cdot \text{Recall}}{\beta^2 \text{Precision} + \text{Recall}}$:
  - If $\beta = 2$ ($F_2$): Weights Recall higher (e.g. Cancer detection, Fraud).
  - If $\beta = 0.5$ ($F_{0.5}$): Weights Precision higher (e.g. Spam filtering where false alarms annoy users).


In [ ]:
# Synthetic Severe Class Imbalance (95% Negative, 5% Positive)
X_imb, y_imb = make_classification(
    n_samples=5000,
    n_features=15,
    n_informative=10,
    weights=[0.95, 0.05],
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb)

# Train a model on imbalanced data
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=["Normal (0)", "Fraud (1)"]))


---
## 2. ROC-AUC vs PR-AUC: The Golden Rule of Model Evaluation

### The Difference:
- **ROC Curve (Receiver Operating Characteristic)**: Plots **True Positive Rate (Recall)** against **False Positive Rate (FPR)**:
  $$\text{FPR} = \frac{\text{FP}}{\text{FP} + \text{TN}}$$
  Because $\text{TN}$ is massive in imbalanced datasets, $\text{FPR}$ remains tiny even when hundreds of false positives occur! This gives a **deceptively optimistic** ROC-AUC score!
- **PR Curve (Precision-Recall Curve)**: Plots **Precision** against **Recall**.
  Because Precision divides by $(\text{TP} + \text{FP})$, it directly exposes False Positives!

> 💡 **Golden Rule**:
> - Use **ROC-AUC** when classes are roughly balanced.
> - Use **PR-AUC (Average Precision)** whenever you have class imbalance (e.g. < 20% positive class)!


In [ ]:
# Compare ROC Curve vs Precision-Recall Curve side-by-side
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_val = auc(fpr, tpr)

prec, rec, _ = precision_recall_curve(y_test, y_prob)
pr_val = average_precision_score(y_test, y_prob)

plt.figure(figsize=(12, 5))

# Subplot 1: ROC Curve
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC-AUC = {roc_val:.3f}')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--', label='Random Guess (0.500)')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve (Looks very optimistic!)')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: Precision-Recall Curve
plt.subplot(1, 2, 2)
plt.plot(rec, prec, color='crimson', lw=2, label=f'PR-AUC = {pr_val:.3f}')
baseline_pr = np.sum(y_test) / len(y_test)
plt.axhline(y=baseline_pr, color='navy', linestyle='--', label=f'Baseline ({baseline_pr:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('PR Curve (Reveals true precision tradeoffs)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## 3. Diagnosing Models with Learning Curves

Learning Curves plot training error and validation error as a function of the training sample size $N$. They diagnose whether your model suffers from **High Bias (Underfitting)** or **High Variance (Overfitting)**.

```text
================ HIGH BIAS (Underfitting) ================
Error
  ^
  |        Training Error (High)
  |-------------------------------------
  |        Validation Error (High)
  |-------------------------------------
  +-------------------------------------> Training Set Size (N)
Diagnosis: Model is too simple.
Remedy: Add more features, increase model complexity, decrease regularization.
Adding more data WILL NOT HELP!

================ HIGH VARIANCE (Overfitting) ================
Error
  ^
  |        Validation Error (High)
  |         |        \----------------------------
  |          Large Gap
  |        /----------------------------
  |       / Training Error (Low)
  +-------------------------------------> Training Set Size (N)
Diagnosis: Model memorized the training set.
Remedy: Add more training data, add regularization (L1/L2), reduce feature count, decrease depth.
```


In [ ]:
# Generate and Plot Learning Curves
train_sizes, train_scores, val_scores = learning_curve(
    estimator=RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42),
    X=X_imb,
    y=y_imb,
    cv=5,
    train_sizes=np.linspace(0.1, 1.0, 6),
    scoring='f1',
    n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', color='navy', label='Training F1 Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='navy')
plt.plot(train_sizes, val_mean, 's-', color='crimson', label='Validation F1 Score (5-fold CV)')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='crimson')

plt.title("Learning Curves (Diagnosing Bias vs Variance)")
plt.xlabel("Training Set Size")
plt.ylabel("F1 Score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 4. Building Leak-Free Production Pipelines with `Pipeline` and `ColumnTransformer`

A frequent critical bug in ML is **Data Leakage**: fitting a scaler, imputer, or encoder on the entire dataset *before* cross-validation or train/test split.
When you scale using the mean/variance of test data, the test data "leaks" into the model, producing artificially high validation scores that crash in production.

Scikit-learn's `Pipeline` guarantees that transformations are fit strictly on training folds!


In [ ]:
# Realistic Tabular Data with Mixed Types (Numeric + Categorical + Missing Values)
raw_df = pd.DataFrame({
    'age': [25, 42, np.nan, 35, 60, 22, 54, np.nan, 45, 33],
    'income': [50000, 120000, 85000, np.nan, 62000, 31000, 140000, 92000, 105000, 48000],
    'education': ['Bachelors', 'Masters', 'PhD', 'Bachelors', 'HighSchool', 'Bachelors', 'Masters', 'PhD', 'Masters', 'HighSchool'],
    'credit_score': [650, 780, 710, 690, np.nan, 580, 810, 740, 760, 620],
    'default': [0, 0, 0, 0, 1, 1, 0, 0, 0, 1]
})

X_raw = raw_df.drop('default', axis=1)
y_raw = raw_df['default']

num_features = ['age', 'income', 'credit_score']
cat_features = ['education']

# 1. Numerical Pipeline: Impute median + Standard Scale
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Categorical Pipeline: Impute most frequent + One-Hot Encode
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Combine in ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

# 4. Full Production Pipeline with Classifier
full_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Fit and predict cleanly with ZERO leakage!
full_pipeline.fit(X_raw, y_raw)
print("Production Pipeline successfully trained!")
sample_inference = full_pipeline.predict_proba(X_raw.iloc[:2])
print(f"Sample Default Probabilities:\n{sample_inference}")


---
## 5. Bayesian Hyperparameter Optimization with Optuna

Why abandon Grid Search and Random Search?
- **Grid Search**: Wastefully checks every combination in an exponential grid ($O(K^D)$), testing unpromising regions over and over.
- **Random Search**: Better than Grid Search, but blind — it does not learn from past evaluations.
- **Bayesian Optimization (Optuna / TPE)**: Fits a probabilistic model $P(\text{score} \mid \text{hyperparameters})$ to balance **Exploration** (trying uncertain regions) and **Exploitation** (focusing around promising configurations).

Let's implement a clean Optuna objective function!


In [ ]:
# Optuna Hyperparameter Optimization Demonstration
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        # Suggest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 20, 150)
        max_depth = trial.suggest_int('max_depth', 2, 10)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        
        clf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42
        )
        clf.fit(X_train, y_train)
        probs = clf.predict_proba(X_test)[:, 1]
        
        # Maximize PR-AUC
        score = average_precision_score(y_test, probs)
        return score

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=15)

    print("=== OPTUNA OPTIMIZATION COMPLETE ===")
    print(f"Best PR-AUC Score: {study.best_value:.4f}")
    print(f"Best Parameters: {study.best_params}")

except ImportError:
    print("Optuna not installed in current environment. Notebook code structure ready for Optuna execution!")


---
## 6. Self-Assessment Practice Exercises & Detailed Solutions

### Exercise 1: Cost-Sensitive Classification
In a medical diagnostic system, a False Negative (missing a malignant tumor) is 50 times worse than a False Positive (calling a benign tumor for a biopsy).
1. Should you optimize for Precision, Recall, or a customized threshold?
2. How do you select the optimal probability classification threshold $p^*$ instead of the default \\\$0.5$?

---
### Exercise 2: Data Leakage Identification
Identify the data leakage flaw in this code snippet:
```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
cv_scores = cross_val_score(RandomForestClassifier(), X_scaled, y, cv=5)
```

---
### Solutions:
<details>
<summary><b>Click to expand Solution 1</b></summary>

1. **Optimize Recall / Cost Matrix**: You should optimize for High Recall and minimize the total expected financial/medical cost:
   $$\text{Total Cost} = C_{FP} \cdot \text{FP} + C_{FN} \cdot \text{FN}$$
2. **Threshold Tuning**: Instead of predicting positive when $P(y=1) > 0.5$, sweep the decision threshold $p \in [0.01, 0.99]$ on validation data and choose the threshold $p^*$ that minimizes the total cost function (which will typically be much lower, e.g. $p^* \approx 0.05-0.15$).
</details>

<details>
<summary><b>Click to expand Solution 2</b></summary>

`scaler.fit_transform(X)` computes the global mean $\mu$ and standard deviation $\sigma$ across the entire dataset $X$ **before** splitting into cross-validation folds. The test fold data points are included in computing $\mu$ and $\sigma$, leaking information about the test distribution into the training process!  
**Fix**: Use a `Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier())])` and pass the pipeline to `cross_val_score()`.
</details>


---
## 7. Summary & Best Practice Checklist
1. **Never evaluate class-imbalanced datasets with Accuracy alone**; always inspect PR-AUC, Confusion Matrix, and F1-macro.
2. **Diagnose Bias vs Variance** using Learning Curves before blindly acquiring more data or increasing model depth.
3. **Always encapsulate preprocessing inside `Pipeline` and `ColumnTransformer`** to eliminate data leakage bugs.
4. **Use Bayesian Optimization (Optuna)** to find optimal hyperparameters efficiently.
